# Evaluación de la configuración entregada

Carga `entrega.json` y mide su precisión sobre los datos de prueba. No hay que editar código, rutas ni parámetros.

**Uso:** elija una GPU T4 y ejecute todas las celdas. La primera vez el entorno se instala y se reinicia; después, ejecute todas las celdas otra vez.

Los datos se toman de `datos_privados.json` en `/content` o se piden con un selector de archivo. Deben tener el formato de `datos_visibles.json`.

In [ ]:
import os

MARCA = "/content/.entorno_listo"
REPO = "https://github.com/Seloga26/ADA2026_2"

if not os.path.exists(MARCA):
    !pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
    !python -m spacy download en_core_web_sm -q
    !git clone -q https://github.com/allenai/open-instruct /content/open-instruct
    !git clone -q {REPO} /content/ADA2026_2

    import io
    import urllib.request
    import zipfile

    NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
    PAQUETES = [
        ("tokenizers", "punkt"),
        ("tokenizers", "punkt_tab"),
        ("taggers", "averaged_perceptron_tagger"),
        ("taggers", "averaged_perceptron_tagger_eng"),
    ]
    for carpeta, nombre in PAQUETES:
        with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

    open(MARCA, "w").close()
    print("Entorno instalado. El entorno se reinicia: vuelva a ejecutar todas las celdas.")
    os.kill(os.getpid(), 9)

%cd /content/ADA2026_2
print("Entorno listo")

In [ ]:
from busqueda_prompt.entrega import cargar_entrega

entrega = cargar_entrega("entrega.json")
config = entrega["config"]
print("grupo:", entrega["grupo"], "· semana:", entrega["semana"])
print("configuración:", config)

In [ ]:
import os
import shutil

from busqueda_prompt.entrega import cargar_instancias

ARCHIVO = "datos_privados.json"
COPIA_COLAB = f"/content/{ARCHIVO}"

if not os.path.exists(ARCHIVO) and os.path.exists(COPIA_COLAB):
    shutil.copy(COPIA_COLAB, ARCHIVO)

if not os.path.exists(ARCHIVO):
    from google.colab import files

    print(f"Seleccione el archivo de datos de prueba ({ARCHIVO}).")
    subidos = files.upload()
    nombre = next(iter(subidos))
    if nombre != ARCHIVO:
        shutil.move(nombre, ARCHIVO)

instancias = cargar_instancias(ARCHIVO)
print(len(instancias), "instancias de prueba")

In [ ]:
from ayudas import cargar_modelo
from oraculo import Oraculo

modelo = cargar_modelo("qwen17b")
oraculo = Oraculo(modelo, instancias, cache=None, lote=16, presupuesto=32_000_000)

r = oraculo.evaluar(config, instancias, semilla=1)
print(f"precisión: {r.precision:.1%}  ({r.n - len(r.trazas)} de {r.n} instancias)")

In [ ]:
from collections import Counter

for familia, veces in Counter(t["violo"] for t in r.trazas).most_common(10):
    print(f"{veces:4}  {familia}")